# Async/Await

Concurrent programming for I/O-bound tasks. AI code often uses this for API calls and web requests.

## Why Async?

Regular (synchronous) code blocks while waiting for I/O:

In [ ]:
import time

def fetch_data_sync(url):
    """Simulate a slow network request."""
    print(f"Fetching {url}...")
    time.sleep(1)  # Block for 1 second
    return f"Data from {url}"

# Sequential execution - 3 seconds total
start = time.time()
result1 = fetch_data_sync("api/users")
result2 = fetch_data_sync("api/posts")
result3 = fetch_data_sync("api/comments")
print(f"Sync time: {time.time() - start:.1f}s")

## Basic Async Syntax

In [ ]:
import asyncio

# 'async def' creates a coroutine function
async def fetch_data_async(url):
    """Simulate async network request."""
    print(f"Fetching {url}...")
    await asyncio.sleep(1)  # Non-blocking sleep
    return f"Data from {url}"

# Calling an async function returns a coroutine object
coro = fetch_data_async("api/test")
print(f"Coroutine object: {coro}")
coro.close()  # Clean up

In [ ]:
# Must run coroutines with asyncio
async def main():
    result = await fetch_data_async("api/users")
    print(f"Got: {result}")

# In Jupyter, use await directly (already has event loop)
# In regular Python, use asyncio.run(main())
await main()

## Running Tasks Concurrently

In [ ]:
import asyncio
import time

async def fetch_data_async(url):
    print(f"Start: {url}")
    await asyncio.sleep(1)
    print(f"Done: {url}")
    return f"Data from {url}"

async def main_concurrent():
    start = time.time()
    
    # Run all three concurrently with gather
    results = await asyncio.gather(
        fetch_data_async("api/users"),
        fetch_data_async("api/posts"),
        fetch_data_async("api/comments"),
    )
    
    print(f"\nAsync time: {time.time() - start:.1f}s")
    print(f"Results: {results}")

await main_concurrent()

## Async Context Managers

In [ ]:
class AsyncResource:
    async def __aenter__(self):
        print("Acquiring resource...")
        await asyncio.sleep(0.1)  # Simulate async setup
        return self
    
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("Releasing resource...")
        await asyncio.sleep(0.1)  # Simulate async cleanup
        return False
    
    async def do_work(self):
        print("Working...")
        await asyncio.sleep(0.1)
        return "done"

async def use_resource():
    async with AsyncResource() as resource:
        result = await resource.do_work()
        print(f"Result: {result}")

await use_resource()

## Async Iteration

In [ ]:
class AsyncCounter:
    def __init__(self, stop):
        self.stop = stop
        self.current = 0
    
    def __aiter__(self):
        return self
    
    async def __anext__(self):
        if self.current >= self.stop:
            raise StopAsyncIteration
        await asyncio.sleep(0.1)  # Simulate async work
        self.current += 1
        return self.current

async def count_async():
    async for num in AsyncCounter(5):
        print(f"Count: {num}")

await count_async()

In [ ]:
# Async generator (simpler)
async def async_range(stop):
    for i in range(stop):
        await asyncio.sleep(0.1)
        yield i

async def use_async_gen():
    async for num in async_range(5):
        print(f"Generated: {num}")

await use_async_gen()

## Task Management

In [ ]:
# Create tasks explicitly
async def slow_operation(name, duration):
    print(f"{name}: starting")
    await asyncio.sleep(duration)
    print(f"{name}: done")
    return f"{name} result"

async def manage_tasks():
    # Create tasks (they start immediately)
    task1 = asyncio.create_task(slow_operation("Task1", 0.5))
    task2 = asyncio.create_task(slow_operation("Task2", 0.3))
    
    # Do other work while tasks run
    print("Tasks are running...")
    await asyncio.sleep(0.1)
    print("Still running...")
    
    # Wait for specific task
    result2 = await task2
    print(f"Task2 finished: {result2}")
    
    # Wait for remaining
    result1 = await task1
    print(f"Task1 finished: {result1}")

await manage_tasks()

In [ ]:
# Handle first completed task
async def race_tasks():
    tasks = [
        asyncio.create_task(slow_operation("Fast", 0.2)),
        asyncio.create_task(slow_operation("Slow", 0.5)),
    ]
    
    # Wait for first to complete
    done, pending = await asyncio.wait(
        tasks, 
        return_when=asyncio.FIRST_COMPLETED
    )
    
    print(f"Completed: {len(done)}, Pending: {len(pending)}")
    
    # Get result from completed task
    for task in done:
        print(f"First result: {task.result()}")
    
    # Cancel pending tasks
    for task in pending:
        task.cancel()

await race_tasks()

## Error Handling

In [ ]:
async def may_fail(should_fail):
    await asyncio.sleep(0.1)
    if should_fail:
        raise ValueError("Something went wrong!")
    return "success"

async def handle_errors():
    # Individual error handling
    try:
        result = await may_fail(True)
    except ValueError as e:
        print(f"Caught: {e}")
    
    # With gather - return_exceptions prevents crash
    results = await asyncio.gather(
        may_fail(False),
        may_fail(True),
        may_fail(False),
        return_exceptions=True
    )
    
    for r in results:
        if isinstance(r, Exception):
            print(f"Error: {r}")
        else:
            print(f"Result: {r}")

await handle_errors()

## Timeouts

In [ ]:
async def slow_api_call():
    await asyncio.sleep(5)  # Very slow!
    return "data"

async def with_timeout():
    try:
        # Wait max 1 second
        result = await asyncio.wait_for(
            slow_api_call(),
            timeout=1.0
        )
        print(f"Got: {result}")
    except asyncio.TimeoutError:
        print("Timed out!")

await with_timeout()

## AI Code Patterns

In [ ]:
# Pattern 1: Concurrent API calls
async def fetch_all(urls):
    """Fetch multiple URLs concurrently."""
    async def fetch_one(url):
        # In real code: async with aiohttp.ClientSession() as session:
        await asyncio.sleep(0.1)  # Simulate request
        return {"url": url, "data": "..."}
    
    return await asyncio.gather(*[fetch_one(url) for url in urls])

urls = ["api/1", "api/2", "api/3"]
results = await fetch_all(urls)
print(f"Fetched {len(results)} URLs")

In [ ]:
# Pattern 2: Rate-limited concurrent requests
async def rate_limited_fetch(urls, max_concurrent=3):
    """Fetch with concurrency limit."""
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def fetch_with_limit(url):
        async with semaphore:
            print(f"Fetching {url}")
            await asyncio.sleep(0.2)
            return f"Data from {url}"
    
    return await asyncio.gather(*[fetch_with_limit(url) for url in urls])

urls = [f"api/{i}" for i in range(6)]
results = await rate_limited_fetch(urls, max_concurrent=2)
print(f"Got {len(results)} results")

In [ ]:
# Pattern 3: Async retry with backoff
async def fetch_with_retry(url, max_retries=3):
    """Retry with exponential backoff."""
    import random
    
    for attempt in range(max_retries):
        try:
            # Simulate occasional failure
            if random.random() < 0.7 and attempt < max_retries - 1:
                raise ConnectionError("Network error")
            await asyncio.sleep(0.1)
            return f"Data from {url}"
        except ConnectionError as e:
            wait = 2 ** attempt * 0.1  # Exponential backoff
            print(f"Attempt {attempt + 1} failed, waiting {wait}s...")
            await asyncio.sleep(wait)
    
    raise Exception(f"Failed after {max_retries} attempts")

result = await fetch_with_retry("api/test")
print(f"Final result: {result}")

## Summary

| Concept | Syntax |
|---------|--------|
| Async function | `async def func():` |
| Await | `await coroutine` |
| Run concurrently | `await asyncio.gather(...)` |
| Create task | `asyncio.create_task(coro)` |
| Async context | `async with resource:` |
| Async iteration | `async for item in items:` |
| Timeout | `await asyncio.wait_for(coro, timeout=x)` |

## Module Complete!

You now understand:
- Dunder methods and how they integrate with Python
- Iterators and generators for lazy evaluation
- Descriptors and properties for attribute control
- Async/await for concurrent programming

Next module: Debugging AI Code!